# CrewAI Demo

CrewAI orchestrates role-based multi-agent teams that mimic real-world human crew dynamics.
The flow is:

LLM -> Agents -> tasks -> Crew -> Process -> kickoff -> crew output

## Required Environment Variables

```
OPENAI_API_KEY
SERPER_API_KEY  # Optional, for web search
```
## Sign up for free SERPER_API_KEY
https://serper.dev/

In [1]:
# Setup
%pip install -q \
    "crewai==1.15.18" \
    "crewai-tools==1.15.18" \
    openai \
    python-dotenv


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


## 1. Load API keys from Google Colab Secrets

Add `OPENAI_API_KEY` and `SERPER_API_KEY` in the Colab **Secrets** panel, and enable notebook access. Never print or commit secret values. Serper is optional; without it, the researcher uses model knowledge rather than live search.

In [2]:
import os
from google.colab import userdata

# Colab Secrets are read at runtime. Strip trailing CR/LF characters
# to avoid invalid HTTP headers when calling the Serper API.
openai_key = userdata.get("OPENAI_API_KEY")
if not openai_key or not openai_key.strip():
    raise ValueError("OPENAI_API_KEY is missing from Colab Secrets")
os.environ["OPENAI_API_KEY"] = openai_key.strip()

try:
    serper_key = userdata.get("SERPER_API_KEY")
except Exception:
    serper_key = None

if serper_key and serper_key.strip():
    os.environ["SERPER_API_KEY"] = serper_key.strip()
else:
    os.environ.pop("SERPER_API_KEY", None)

print("OpenAI API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Serper API key loaded:", bool(os.getenv("SERPER_API_KEY")))

OpenAI API key loaded: True
Serper API key loaded: True


## What We're Building

A **Content Marketing Team** — three agents (researcher, writer, editor) collaborating to produce a blog post about AI-powered content marketing.

In [4]:
import os
import yaml
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool
# LLM configure
llm = LLM(model="gpt-4o", temperature=0.7)

web_search_tool = None
try:
    if os.getenv("SERPER_API_KEY"):
        web_search_tool = SerperDevTool()
        print("SerperDevTool initialized")
    else:
        print("SERPER_API_KEY unavailable; no live web search")
except Exception as exc:
    print(f"SerperDevTool initialization failed: {type(exc).__name__}: {exc}")

SerperDevTool initialized


## Agent Definition

Agents are role-based team members with a **role**, **goal**, **backstory**, and **tools**.

In [5]:
agents_yaml_content = """
content_researcher:
  role: Senior Content Researcher
  goal: Research and gather comprehensive information on given topics to support content creation
  backstory: Experienced researcher with 10+ years in digital marketing, skilled at finding trends and actionable insights.
  tools:
  - web_search
  verbose: true
  allow_delegation: false

content_writer:
  role: Senior Content Writer
  goal: Create engaging, high-quality content based on research and brand guidelines
  backstory: Seasoned content writer who transforms research into compelling marketing narratives.
  tools: []
  verbose: true
  allow_delegation: false

content_editor:
  role: Senior Content Editor
  goal: Review, refine, and ensure content quality meets brand standards and objectives
  backstory: Experienced editor with a keen eye for detail, brand consistency, and audience engagement.
  tools: []
  verbose: true
  allow_delegation: true
"""

agents_config = yaml.safe_load(agents_yaml_content.strip())

agents_config

{'content_researcher': {'role': 'Senior Content Researcher',
  'goal': 'Research and gather comprehensive information on given topics to support content creation',
  'backstory': 'Experienced researcher with 10+ years in digital marketing, skilled at finding trends and actionable insights.',
  'tools': ['web_search'],
  'verbose': True,
  'allow_delegation': False},
 'content_writer': {'role': 'Senior Content Writer',
  'goal': 'Create engaging, high-quality content based on research and brand guidelines',
  'backstory': 'Seasoned content writer who transforms research into compelling marketing narratives.',
  'tools': [],
  'verbose': True,
  'allow_delegation': False},
 'content_editor': {'role': 'Senior Content Editor',
  'goal': 'Review, refine, and ensure content quality meets brand standards and objectives',
  'backstory': 'Experienced editor with a keen eye for detail, brand consistency, and audience engagement.',
  'tools': [],
  'verbose': True,
  'allow_delegation': True}}

In [6]:
tools_map = {"web_search": web_search_tool}

agents = {}
for agent_name, config in agents_config.items():
    configured_tools = config.get("tools", [])
    unknown_tools = [name for name in configured_tools if name not in tools_map]
    if unknown_tools:
        raise ValueError(f"Unknown tools for {agent_name}: {unknown_tools}")
    agent_tools = [tools_map[name] for name in configured_tools if tools_map[name] is not None]

    agents[agent_name] = Agent(
        role=config["role"],
        goal=config["goal"],
        backstory=config["backstory"],
        tools=agent_tools,
        verbose=config.get("verbose", True),
        allow_delegation=config.get("allow_delegation", False),
        llm=llm,
        max_iter=5,
    )

researcher = agents["content_researcher"]
writer = agents["content_writer"]
editor = agents["content_editor"]

for name, agent in agents.items():
    print(f"{agent.role}: {len(agent.tools)} tool(s), delegation={'yes' if agent.allow_delegation else 'no'}")

Senior Content Researcher: 1 tool(s), delegation=no
Senior Content Writer: 0 tool(s), delegation=no
Senior Content Editor: 0 tool(s), delegation=yes


### Agent execution loop and delegation

Within a task, an Agent can call its LLM, decide to invoke a tool, inspect the tool response, and repeat before producing a final answer. `max_iter=5` limits agent iterations; it does **not** define five tasks. Only the Researcher has Serper search configured. `allow_delegation=True` permits the Editor to ask another agent for help, but does not guarantee it will happen or alter the sequential task list.

## Task Definition

Tasks are structured work items with a **description**, **expected output**, **assigned agent**, and optional **context** (dependencies on other tasks).

In [7]:
tasks_yaml_content = """
research_task:
  description: Research the latest trends and best practices in AI-powered content marketing for 2026. Focus on practical applications, case studies, and emerging technologies.
  agent: content_researcher
  expected_output: A comprehensive research report with key trends, statistics, and actionable insights.

writing_task:
  description: Create an engaging blog post about AI-powered content marketing based on the research findings. Target audience is marketing professionals.
  agent: content_writer
  expected_output: A well-structured 1200-1500 word blog post with introduction, key sections, examples, and conclusion.
  context:
  - research_task

editing_task:
  description: Review and refine the blog post for clarity, engagement, brand consistency, and SEO optimization.
  agent: content_editor
  expected_output: A polished, publication-ready blog post with improved flow, grammar, and SEO.
  context:
  - research_task
  - writing_task
"""

tasks_config = yaml.safe_load(tasks_yaml_content.strip())

for task_name, config in tasks_config.items():
    deps = config.get('context', ['None'])
    print(f"{task_name}: agent={config['agent']}, depends_on={', '.join(deps)}")

research_task: agent=content_researcher, depends_on=None
writing_task: agent=content_writer, depends_on=research_task
editing_task: agent=content_editor, depends_on=research_task, writing_task


In [8]:
tasks = {}

for task_name, config in tasks_config.items():
    dependencies = config.get("context", [])
    missing = [dep for dep in dependencies if dep not in tasks]
    if missing:
        raise ValueError(
            f"Task {task_name!r} has missing or out-of-order dependencies: {missing}"
        )
    agent_name = config["agent"]
    if agent_name not in agents:
        raise ValueError(f"Unknown agent {agent_name!r} in task {task_name!r}")

    tasks[task_name] = Task(
        description=config["description"],
        agent=agents[agent_name],
        expected_output=config["expected_output"],
        context=[tasks[dep] for dep in dependencies] or None,
    )

research_task = tasks["research_task"]
writing_task = tasks["writing_task"]
editing_task = tasks["editing_task"]

for i, (name, task) in enumerate(tasks.items(), 1):
    deps = tasks_config[name].get("context", [])
    print(f"{i}. {name} -> {task.agent.role} (depends: {', '.join(deps) or 'none'})")

1. research_task -> Senior Content Researcher (depends: none)
2. writing_task -> Senior Content Writer (depends: research_task)
3. editing_task -> Senior Content Editor (depends: research_task, writing_task)


### Task context vs delegation

| Mechanism | What it does | Example |
|---|---|---|
| `Task.context` | Passes outputs of specified prior tasks | Writer reads the research report |
| `allow_delegation` | Allows an agent to request another agent's assistance while working | Editor asks Researcher to verify a claim |

`Process.sequential` schedules the three tasks in order.

## Crew Orchestration

The **Crew** orchestrates agents and tasks. Process types: **sequential**, **hierarchical**

In [9]:
content_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print("Crew initialized: 3 agents, 3 tasks, sequential process")

Crew initialized: 3 agents, 3 tasks, sequential process


## Execute Crew

In [10]:
# In Google Colab, top-level await is supported.
# The asynchronous API does not make sequential tasks run in parallel.
result = await content_crew.kickoff_async()

print("\n===== FINAL DELIVERABLE =====")
print(result.raw)

print("\n===== TASK OUTPUTS =====")
for i, output in enumerate(result.tasks_output, 1):
    print(f"\n--- Task {i}: {output.agent} ---")
    print(output.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.18                                                                                       │
│  Latest version:  1.15.30                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: edabd291-fb67-4903-b310-1f18abbe8cd3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest trends and best practices in AI-powered content marketing for 2026. Focus on         │
│  practical applications, case studies, and emerging technologies.                                               │
│  ID: 6c455e55-263f-48fe-8e7b-fb0b892ff3cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Content Researcher                                                                               │
│                                                                                                                 │
│  Task: Research the latest trends and best practices in AI-powered content marketing for 2026. Focus on         │
│  practical applications, case studies, and emerging technologies.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'AI-powered content marketing trends 2026'}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Content Editor                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Title: The Future of Content Marketing: Unveiling AI-Powered Trends for 2026**                               │
│                                                                                                                 │
│  **Meta Description:**                                                                                          │
│  Discover the latest AI-powered content marketing trends for 2026, including AI-driven personalization,         │
│  generative AI, and more. Stay ahead with these innovative strategies.                                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  The landscape of content marketing is evolving at a breakneck pace, with AI technologies at the forefront of   │
│  this transformation. As we look toward 2026, understanding how AI-powered trends will shape the industry is    │
│  crucial for marketers aiming to stay competitive. Here, we delve into the top AI-driven content marketing      │
│  trends set to redefine the way brands connect with their audiences.                                            │
│                                                                                                                 │
│  ### 1. AI-Driven Personalization                                                                               │
│                                                                                                                 │
│  As consumers demand more personalized experiences, AI-driven personalization is becoming a critical component  │
│  of content marketing strategies. By leveraging AI, marketers can analyze vast amounts of data to deliver       │
│  tailored content that resonates with individual preferences and behaviors. This trend not only enhances user   │
│  engagement but also drives higher conversion rates.                                                            │
│                                                                                                                 │
│  **SEO Tip:** Use keywords such as "AI personalization," "tailored content," and "consumer engagement" to       │
│  optimize for search engines.                                                                                   │
│                                                                                                                 │
│  ### 2. Generative AI in Content Creation                                                                       │
│                                                                                                                 │
│  Generative AI is revolutionizing content creation by enabling the production of high-quality, original         │
│  content at scale. This technology can generate articles, videos, and even interactive content, reducing the    │
│  time and resources required for content development. For marketers, this means more efficient content          │
│  strategies and the ability to maintain a consistent brand voice across all platforms.                          │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review and refine the blog post for clarity, engagement, brand consistency, and SEO optimization.        │
│  Agent: Senior Content Editor                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL DELIVERABLE =====
**Title: The Future of Content Marketing: Unveiling AI-Powered Trends for 2026**

**Meta Description:**
Discover the latest AI-powered content marketing trends for 2026, including AI-driven personalization, generative AI, and more. Stay ahead with these innovative strategies.

---

The landscape of content marketing is evolving at a breakneck pace, with AI technologies at the forefront of this transformation. As we look toward 2026, understanding how AI-powered trends will shape the industry is crucial for marketers aiming to stay competitive. Here, we delve into the top AI-driven content marketing trends set to redefine the way brands connect with their audiences.

### 1. AI-Driven Personalization

As consumers demand more personalized experiences, AI-driven personalization is becoming a critical component of content marketing strategies. By leveraging AI, marketers can analyze vast amounts of data to deliver tailored content that resonates with individu

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: edabd291-fb67-4903-b310-1f18abbe8cd3                                                                       │
│  Final Output: **Title: The Future of Content Marketing: Unveiling AI-Powered Trends for 2026**                 │
│                                                                                                                 │
│  **Meta Description:**                                                                                          │
│  Discover the latest AI-powered content marketing trends for 2026, including AI-driven personalization,         │
│  generative AI, and more. Stay ahead with these innovative strategies.                                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  The landscape of content marketing is evolving at a breakneck pace, with AI technologies at the forefront of   │
│  this transformation. As we look toward 2026, understanding how AI-powered trends will shape the industry is    │
│  crucial for marketers aiming to stay competitive. Here, we delve into the top AI-driven content marketing      │
│  trends set to redefine the way brands connect with their audiences.                                            │
│                                                                                                                 │
│  ### 1. AI-Driven Personalization                                                                               │
│                                                                                                                 │
│  As consumers demand more personalized experiences, AI-driven personalization is becoming a critical component  │
│  of content marketing strategies. By leveraging AI, marketers can analyze vast amounts of data to deliver       │
│  tailored content that resonates with individual preferences and behaviors. This trend not only enhances user   │
│  engagement but also drives higher conversion rates.                                                            │
│                                                                                                                 │
│  **SEO Tip:** Use keywords such as "AI personalization," "tailored content," and "consumer engagement" to       │
│  optimize for search engines.                                                                                   │
│                                                                                                                 │
│  ### 2. Generative AI in Content Creation                                                                       │
│                                                                                                                 │
│  Generative AI is revolutionizing content creation by enabling the production of high-quality, original         │
│  content at scale. This technology can generate articles, videos, and even interactive content, reducing the    │
│  time and resources required for content development. For marketers, this means more efficient content          │
│  strategies and the ability to maintain a consistent brand voice across all platforms.                          │
│                                                       

## Summary

CrewAI provides role-based multi-agent orchestration with:
- **Agents**: Team members defined by role, goal, backstory, tools, and optional delegation
- **Tasks**: Work items with expected outputs and explicit context dependencies
- **Crew**: Runs the tasks using **sequential** or **hierarchical** processes
- **YAML config**: Keeps role and task definitions separate from Python object creation
- **Delegation vs context**: Delegation lets an agent request help; context passes completed task outputs. Neither is the same as a handoff.
- **Crew vs Flow**: This notebook demonstrates a Crew, not a Flow. A Flow can orchestrate Crews and conditional workflow branches.

### Resources
- [CrewAI Documentation](https://docs.crewai.com/)
- [CrewAI GitHub](https://github.com/crewAIInc/crewAI)